# Module 5: Deep Reinforcement Learning
## Notebook 2: DQN Improvements — Double DQN

**Learning Objectives**

By completing this notebook, you will:
1. Understand and quantify the Q-value overestimation bias in vanilla DQN
2. Implement Double DQN (DDQN) — a minimal two-line change that eliminates overestimation
3. Plot Q-value estimates vs. true values to make overestimation concrete
4. Compare DQN and DDQN learning curves on CartPole-v1 with multiple runs

**Prerequisites**
- DQN from scratch (Notebook 1) — replay buffer, target network, training loop
- PyTorch basics

**Estimated Time: 13 minutes**

---

### The Overestimation Problem

Vanilla DQN uses the same network to **select** the best action and **evaluate** its value:

$$\text{DQN target: } y = R + \gamma \max_{a'} Q_{\theta^-}(S', a')$$

The $\max$ operator is biased: if Q-estimates have any noise (they always do), then $\mathbb{E}[\max_a Q(s, a)] \ge \max_a \mathbb{E}[Q(s, a)]$ by Jensen's inequality. The network systematically overestimates values.

**Double DQN** decouples action selection from action evaluation:

$$\text{DDQN target: } y = R + \gamma Q_{\theta^-}\!\left(S',\, \underbrace{\arg\max_{a'} Q_{\theta}(S', a')}_{\text{online net selects}}\right)$$

The **online network** selects which action is best; the **target network** evaluates its value. These two networks have different noise patterns, so they don't compound each other's overestimates.

In [ ]:
learning_objectives([
    "By completing this notebook, you will:",
    "- DQN from scratch (Notebook 1) — replay buffer, target network, training loop",
    "PyTorch basics",
    "---",
    "the best action and **evaluate** its value:",
    "decouples action selection from action evaluation:",
    "selects which action is best; the **target network** evaluates its value. These two networks have different noise patterns, so they don't compound each other's overestimates."
])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import all dependencies and set reproducibility seeds
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque
import random

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

DEVICE    = torch.device('cpu')
OBS_DIM   = 4
N_ACTIONS = 2

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print("Setup complete.")

## 1. Core Components (Reproduced from Notebook 1)

We reproduce the ReplayBuffer and QNetwork here so this notebook is self-contained.

In [ ]:
# Purpose: Reproduce ReplayBuffer and QNetwork (self-contained notebook)

class ReplayBuffer:
    """Fixed-size circular experience replay buffer."""

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states),      dtype=torch.float32).to(DEVICE),
            torch.tensor(actions,               dtype=torch.long).to(DEVICE),
            torch.tensor(rewards,               dtype=torch.float32).to(DEVICE),
            torch.tensor(np.array(next_states), dtype=torch.float32).to(DEVICE),
            torch.tensor(dones,                 dtype=torch.float32).to(DEVICE),
        )

    def __len__(self):
        return len(self.buffer)


class QNetwork(nn.Module):
    """3-layer MLP: obs_dim -> 128 -> 128 -> n_actions."""

    def __init__(self, obs_dim, n_actions, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        )

    def forward(self, x):
        return self.net(x)


print("ReplayBuffer and QNetwork defined.")

## 2. DQN and Double DQN Agents

The only difference between DQN and DDQN is how the target is computed:

- **DQN**: `target = r + gamma * target_net(s').max(dim=1).values`
- **DDQN**: `best_actions = online_net(s').argmax(dim=1)` then `target = r + gamma * target_net(s').gather(1, best_actions)`

We implement this as a single agent class with a `double_dqn` flag.

In [ ]:
# Purpose: Implement a unified DQN/DDQN agent with a single flag to toggle algorithm
# Key Concept: DDQN differs from DQN only in which network selects the target action

class DQNDDQNAgent:
    """
    DQN or Double DQN agent.

    Parameters
    ----------
    double_dqn : bool — if True, use Double DQN target; if False, use vanilla DQN
    """

    def __init__(self, obs_dim, n_actions, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
                 buffer_capacity=10_000, batch_size=64, target_update_freq=100,
                 double_dqn=False):

        self.n_actions   = n_actions
        self.gamma       = gamma
        self.epsilon     = epsilon_start
        self.epsilon_end = epsilon_end
        self.eps_decay   = epsilon_decay
        self.batch_size  = batch_size
        self.target_freq = target_update_freq
        self.double_dqn  = double_dqn
        self.steps       = 0

        self.q_net    = QNetwork(obs_dim, n_actions).to(DEVICE)
        self.q_target = QNetwork(obs_dim, n_actions).to(DEVICE)
        self.q_target.load_state_dict(self.q_net.state_dict())
        self.q_target.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_capacity)

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            obs_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            return int(self.q_net(obs_t).argmax(dim=1).item())

    def compute_target(self, rewards, next_states, dones):
        """
        Compute TD targets.

        DQN target:   r + gamma * max_a Q_target(s', a)
        DDQN target:  r + gamma * Q_target(s', argmax_a Q_online(s', a))

        Returns
        -------
        targets : torch.Tensor, shape (batch_size,)
        """
        with torch.no_grad():
            if self.double_dqn:
                # Step A: Use ONLINE net to select the best action
                best_actions = self.q_net(next_states).argmax(dim=1, keepdim=True)
                # Step B: Use TARGET net to evaluate that action's value
                next_q = self.q_target(next_states).gather(1, best_actions).squeeze(1)
            else:
                # DQN: use target net for both selection and evaluation
                next_q = self.q_target(next_states).max(dim=1).values

            targets = rewards + self.gamma * next_q * (1.0 - dones)
        return targets

    def update(self, state, action, reward, next_state, done):
        self.buffer.push(state, action, reward, next_state, done)
        self.steps += 1
        self.epsilon = max(self.epsilon_end, self.epsilon * self.eps_decay)

        if len(self.buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)

        # Current Q-values for selected actions
        q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Targets
        targets = self.compute_target(rewards, next_states, dones)

        loss = F.mse_loss(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        if self.steps % self.target_freq == 0:
            self.q_target.load_state_dict(self.q_net.state_dict())

        return loss.item()


print("DQNDDQNAgent defined.")
# Verify the flag works
ag_dqn  = DQNDDQNAgent(OBS_DIM, N_ACTIONS, double_dqn=False)
ag_ddqn = DQNDDQNAgent(OBS_DIM, N_ACTIONS, double_dqn=True)
print(f"DQN agent double_dqn={ag_dqn.double_dqn}, DDQN agent double_dqn={ag_ddqn.double_dqn}")

## 3. Training with Q-Value Tracking

To make the overestimation bias concrete, we track the agent's Q-value estimates at each episode start and compare them to the true Monte Carlo returns from that starting state.

In [ ]:
# Purpose: Training loop that records Q-value estimates AND true MC returns for comparison
# Key Concept: Comparing estimated Q-values to true returns reveals overestimation

def estimate_true_value(env, agent, obs, gamma=0.99, n_rollouts=5):
    """
    Estimate the true value of a starting state via Monte Carlo rollouts
    using the current greedy policy.

    Returns
    -------
    float — average discounted return over n_rollouts
    """
    returns = []
    for _ in range(n_rollouts):
        state = obs.copy()
        G = 0.0
        discount = 1.0
        env2 = gym.make('CartPole-v1')
        s2, _ = env2.reset()
        s2 = state   # set to the actual start state (env doesn't support this directly)

        # Run a single episode from the current env state using greedy policy
        # We can't set arbitrary initial states in CartPole, so we use the
        # episode-start observation as the reference state
        done2 = False
        max_steps = 500
        for _ in range(max_steps):
            a = agent.select_action(s2)
            ns2, r2, t2, tr2, _ = env2.step(a)
            done2 = t2 or tr2
            G += discount * r2
            discount *= gamma
            s2 = ns2
            if done2:
                break

        returns.append(G)
        env2.close()

    return float(np.mean(returns))


def train_with_q_tracking(double_dqn, n_episodes=300, seed=42, track_every=10):
    """
    Train DQN or DDQN and track Q-value estimates vs. true MC values.

    Returns
    -------
    episode_rewards    : list
    tracked_episodes   : list — episode indices where tracking occurred
    estimated_q_vals   : list — agent's estimated max Q-value at episode start
    true_mc_vals       : list — Monte Carlo return from episode-start state
    """
    env   = gym.make('CartPole-v1')
    agent = DQNDDQNAgent(
        OBS_DIM, N_ACTIONS,
        lr=1e-3, gamma=0.99,
        epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
        buffer_capacity=10_000, batch_size=64, target_update_freq=100,
        double_dqn=double_dqn,
    )

    episode_rewards  = []
    tracked_episodes = []
    estimated_q_vals = []
    true_mc_vals     = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)

        # Track Q-estimates every 'track_every' episodes
        if ep % track_every == 0:
            with torch.no_grad():
                obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                est_q = agent.q_net(obs_t).max().item()
            true_val = estimate_true_value(env, agent, obs, n_rollouts=3)

            tracked_episodes.append(ep)
            estimated_q_vals.append(est_q)
            true_mc_vals.append(true_val)

        total_reward = 0.0
        done = False

        while not done:
            action = agent.select_action(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.update(obs, action, reward, next_obs, done)
            total_reward += reward
            obs = next_obs

        episode_rewards.append(total_reward)

    env.close()
    return episode_rewards, tracked_episodes, estimated_q_vals, true_mc_vals


print("Training DQN with Q-tracking...")
r_dqn, eps_dqn, eq_dqn, tv_dqn = train_with_q_tracking(double_dqn=False, n_episodes=300)
print(f"DQN final mean-100: {np.mean(r_dqn[-100:]):.1f}")

print("Training DDQN with Q-tracking...")
r_ddqn, eps_ddqn, eq_ddqn, tv_ddqn = train_with_q_tracking(double_dqn=True, n_episodes=300)
print(f"DDQN final mean-100: {np.mean(r_ddqn[-100:]):.1f}")

## 4. Visualize Overestimation Bias

In [ ]:
# Purpose: Plot Q-value estimates vs. true MC returns for DQN and DDQN
# Key Concept: The gap between estimated Q and true value reveals overestimation

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (est_q, true_v, eps_tracked, r_ep, title) in zip(axes, [
    (eq_dqn,  tv_dqn,  eps_dqn,  r_dqn,  'Vanilla DQN'),
    (eq_ddqn, tv_ddqn, eps_ddqn, r_ddqn, 'Double DQN'),
]):
    ax2 = ax.twinx()

    # Estimated vs. true Q-values
    ax.plot(eps_tracked, est_q,  'o-', color='tomato',    markersize=4,
            linewidth=1.5, label='Estimated max Q(s,a)')
    ax.plot(eps_tracked, true_v, 's-', color='steelblue', markersize=4,
            linewidth=1.5, label='True MC return')

    # Shade overestimation region
    ax.fill_between(eps_tracked,
                    true_v, est_q,
                    where=[e > t for e, t in zip(est_q, true_v)],
                    alpha=0.15, color='tomato', label='Overestimation')

    # Episode rewards on secondary axis
    ax2.plot(r_ep, color='gray', alpha=0.3, linewidth=0.8)
    ax2.set_ylabel('Episode Reward', color='gray', fontsize=10)

    ax.set_xlabel('Episode', fontsize=11)
    ax.set_ylabel('Value Estimate', fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9, loc='upper left')

plt.suptitle('Q-Value Overestimation: DQN vs. Double DQN', fontsize=14)
plt.tight_layout()
plt.savefig('../resources/ddqn_overestimation.png', dpi=120, bbox_inches='tight')
plt.show()

# Quantify average overestimation in second half of training
n_half = len(eps_dqn) // 2
dqn_bias  = np.mean(np.array(eq_dqn[n_half:])  - np.array(tv_dqn[n_half:]))
ddqn_bias = np.mean(np.array(eq_ddqn[n_half:]) - np.array(tv_ddqn[n_half:]))
print(f"DQN  overestimation (avg second half): {dqn_bias:+.2f}")
print(f"DDQN overestimation (avg second half): {ddqn_bias:+.2f}")

## 5. Multi-Run Learning Curve Comparison

In [ ]:
# Purpose: Statistically compare DQN and DDQN over multiple runs
# Key Concept: Multiple runs reveal whether the improvement is consistent or lucky

def train_simple(double_dqn, n_episodes=300, seed=42):
    """Lightweight training (no Q-tracking) for multi-run comparison."""
    env   = gym.make('CartPole-v1')
    agent = DQNDDQNAgent(
        OBS_DIM, N_ACTIONS,
        lr=1e-3, gamma=0.99,
        epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
        buffer_capacity=10_000, batch_size=64, target_update_freq=100,
        double_dqn=double_dqn,
    )
    rewards = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total  = 0.0
        done   = False
        while not done:
            action = agent.select_action(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.update(obs, action, reward, next_obs, done)
            total += reward
            obs = next_obs
        rewards.append(total)

    env.close()
    return np.array(rewards)


N_RUNS     = 8     # enough to show trends without long training
N_EPISODES = 300

dqn_runs  = np.zeros((N_RUNS, N_EPISODES))
ddqn_runs = np.zeros((N_RUNS, N_EPISODES))

print("Running multi-run comparison...")
for run in range(N_RUNS):
    dqn_runs[run]  = train_simple(double_dqn=False, n_episodes=N_EPISODES, seed=run*7)
    ddqn_runs[run] = train_simple(double_dqn=True,  n_episodes=N_EPISODES, seed=run*7)
    print(f"  Run {run+1}/{N_RUNS}: DQN={dqn_runs[run,-50:].mean():.0f}, DDQN={ddqn_runs[run,-50:].mean():.0f}")

print("Complete.")

In [ ]:
# Purpose: Plot mean ± std learning curves for DQN vs DDQN
# Key Concept: DDQN typically converges faster and/or more stably

def smooth_2d(arr, window=20):
    """Apply moving average to each row of a 2D array."""
    kernel = np.ones(window) / window
    return np.array([np.convolve(row, kernel, mode='valid') for row in arr])


dqn_s  = smooth_2d(dqn_runs,  window=20)
ddqn_s = smooth_2d(ddqn_runs, window=20)
x      = np.arange(dqn_s.shape[1])

fig, ax = plt.subplots(figsize=(13, 6))

for data, label, color in [
    (dqn_s,  'DQN',         'tomato'),
    (ddqn_s, 'Double DQN',  'steelblue'),
]:
    mean = data.mean(axis=0)
    std  = data.std(axis=0)
    ax.plot(x, mean, color=color, linewidth=2.5,
            label=f'{label} (mean {N_RUNS} runs)')
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18)

ax.axhline(475, color='green', linestyle='--', linewidth=1.5, label='Solved (475)')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Reward (smoothed)', fontsize=12)
ax.set_title(f'DQN vs. Double DQN on CartPole-v1 ({N_RUNS} runs, shaded = ±1 std)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(-10, 510)
plt.tight_layout()
plt.savefig('../resources/dqn_vs_ddqn_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"DQN  final mean-100 (across {N_RUNS} runs): {dqn_runs[:, -100:].mean():.1f} ± {dqn_runs[:, -100:].mean(axis=1).std():.1f}")
print(f"DDQN final mean-100 (across {N_RUNS} runs): {ddqn_runs[:, -100:].mean():.1f} ± {ddqn_runs[:, -100:].mean(axis=1).std():.1f}")

### Exercise 5.2.1: Implement Soft Target Updates

**Task:** Instead of copying the entire target network every N steps (hard update), implement **soft updates** (also called Polyak averaging):

$$\theta^- \leftarrow \tau \theta + (1-\tau) \theta^-$$

where $\tau \in (0, 1]$ is a small mixing coefficient (typically 0.01). Modify `DQNDDQNAgent` to use soft updates, train Double DQN with tau=0.01 for 300 episodes, and compare to the hard-update version.

**Expected Output:** Learning curves for hard and soft target update DDQN. Soft updates often converge more smoothly.

<details>
<summary>Hint 1</summary>
In the update method, replace the periodic hard copy with: for p, p_target in zip(q_net.parameters(), q_target.parameters()): p_target.data.copy_(tau * p.data + (1 - tau) * p_target.data)
</details>

<details>
<summary>Hint 2</summary>
With soft updates, you update the target network every step (not every N steps). Set tau=0.01 and remove the target_update_freq logic.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

class DDQNSoftUpdate:
    """
    Double DQN with Polyak (soft) target network updates.
    theta_target = tau * theta_online + (1 - tau) * theta_target
    """

    def __init__(self, obs_dim, n_actions, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
                 buffer_capacity=10_000, batch_size=64, tau=0.01):

        self.n_actions   = n_actions
        self.gamma       = gamma
        self.epsilon     = epsilon_start
        self.epsilon_end = epsilon_end
        self.eps_decay   = epsilon_decay
        self.batch_size  = batch_size
        self.tau         = tau

        self.q_net    = QNetwork(obs_dim, n_actions).to(DEVICE)
        self.q_target = QNetwork(obs_dim, n_actions).to(DEVICE)
        self.q_target.load_state_dict(self.q_net.state_dict())
        self.q_target.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_capacity)

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            obs_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            return int(self.q_net(obs_t).argmax(dim=1).item())

    def soft_update_target(self):
        """Polyak averaging: target = tau * online + (1 - tau) * target."""
        for p_online, p_target in zip(self.q_net.parameters(), self.q_target.parameters()):
            p_target.data.copy_(self.tau * p_online.data + (1.0 - self.tau) * p_target.data)

    def update(self, state, action, reward, next_state, done):
        self.buffer.push(state, action, reward, next_state, done)
        self.epsilon = max(self.epsilon_end, self.epsilon * self.eps_decay)

        if len(self.buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)

        q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            # Double DQN: online selects, target evaluates
            best_actions = self.q_net(next_states).argmax(dim=1, keepdim=True)
            next_q       = self.q_target(next_states).gather(1, best_actions).squeeze(1)
            targets      = rewards + self.gamma * next_q * (1.0 - dones)

        loss = F.mse_loss(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        # Soft update every step
        self.soft_update_target()

        return loss.item()


def train_soft(n_episodes=300, seed=42):
    env   = gym.make('CartPole-v1')
    agent = DDQNSoftUpdate(
        OBS_DIM, N_ACTIONS, lr=1e-3, gamma=0.99,
        epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
        buffer_capacity=10_000, batch_size=64, tau=0.01,
    )
    rewards = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total  = 0.0
        done   = False
        while not done:
            action = agent.select_action(obs)
            next_obs, r, t, tr, _ = env.step(action)
            done = t or tr
            agent.update(obs, action, r, next_obs, done)
            total += r
            obs = next_obs
        rewards.append(total)
    env.close()
    return np.array(rewards)


# Run both hard and soft update DDQN for 5 seeds
N_RUNS_EX = 5
hard_update_runs = np.zeros((N_RUNS_EX, 300))
soft_update_runs = np.zeros((N_RUNS_EX, 300))

for run in range(N_RUNS_EX):
    hard_update_runs[run] = train_simple(double_dqn=True,  n_episodes=300, seed=run*13)
    soft_update_runs[run] = train_soft(n_episodes=300, seed=run*13)
    print(f"Run {run+1}: Hard={hard_update_runs[run,-50:].mean():.0f}, Soft={soft_update_runs[run,-50:].mean():.0f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
for data, label, color in [
    (hard_update_runs, 'DDQN Hard Updates (every 100 steps)', 'steelblue'),
    (soft_update_runs, 'DDQN Soft Updates (tau=0.01)',         'forestgreen'),
]:
    sm  = smooth_2d(data, window=20)
    m   = sm.mean(axis=0)
    s   = sm.std(axis=0)
    ax.plot(m, color=color, linewidth=2, label=label)
    ax.fill_between(np.arange(len(m)), m-s, m+s, color=color, alpha=0.2)

ax.axhline(475, color='green', linestyle='--', linewidth=1.5)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Reward (smoothed)', fontsize=12)
ax.set_title(f'Hard vs. Soft Target Updates in DDQN ({N_RUNS_EX} runs)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(-10, 510)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_521():
    assert hard_update_runs.shape == (N_RUNS_EX, 300), \
        f"hard_update_runs shape {hard_update_runs.shape} != ({N_RUNS_EX}, 300)"
    assert soft_update_runs.shape == (N_RUNS_EX, 300), \
        f"soft_update_runs shape {soft_update_runs.shape} != ({N_RUNS_EX}, 300)"

    # Soft update agent should use tau attribute
    agent_test = DDQNSoftUpdate(OBS_DIM, N_ACTIONS, tau=0.01)
    assert hasattr(agent_test, 'tau'), "DDQNSoftUpdate should have a 'tau' attribute"
    assert agent_test.tau == 0.01, f"Expected tau=0.01, got {agent_test.tau}"

    # Verify soft update changes target network weights
    old_target_w = agent_test.q_target.net[0].weight.data.clone()
    agent_test.q_net.net[0].weight.data.fill_(1.0)  # force a change
    agent_test.soft_update_target()
    new_target_w = agent_test.q_target.net[0].weight.data
    assert not torch.allclose(old_target_w, new_target_w), \
        "Target weights should change after soft_update_target()"

    # Both should achieve reasonable performance
    hard_final = hard_update_runs[:, -50:].mean()
    soft_final = soft_update_runs[:, -50:].mean()
    assert hard_final > 50, f"Hard updates should reach mean > 50, got {hard_final:.1f}"
    assert soft_final > 50, f"Soft updates should reach mean > 50, got {soft_final:.1f}"

    print("Exercise 5.2.1 passed!")
    print(f"Hard updates final: {hard_final:.1f}, Soft updates final: {soft_final:.1f}")

test_exercise_521()

### Exercise 5.2.2: Overestimation as a Function of Network Noise

**Task:** Overestimation is caused by noise in Q-estimates. To demonstrate this analytically, create a synthetic experiment: define a set of 10 "states" with true Q-values Q_true = [1, 2, 3, ..., 10]. Add Gaussian noise (std=sigma) and compute the expected max of noisy estimates vs. max of true values. Plot overestimation = E[max(Q + noise)] - max(Q) for sigma in [0, 0.5, 1.0, 2.0, 5.0].

**Expected Output:** A line plot showing overestimation increasing with noise level, proving the bias comes from the max operator on noisy estimates.

<details>
<summary>Hint 1</summary>
For each sigma, run 10000 trials: draw noise = np.random.normal(0, sigma, 10), compute max(Q_true + noise), average over trials. Compare to max(Q_true) = 10.
</details>

<details>
<summary>Hint 2</summary>
This is Jensen's inequality in action: E[max(Q + noise)] >= max(E[Q + noise]) = max(Q).
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

Q_TRUE   = np.arange(1, 11, dtype=float)   # true Q-values: [1, 2, ..., 10]
TRUE_MAX = Q_TRUE.max()                    # = 10.0
N_TRIALS = 10_000
sigmas   = [0.0, 0.5, 1.0, 2.0, 5.0]

overestimations = []

for sigma in sigmas:
    if sigma == 0.0:
        overestimation = 0.0
    else:
        # Add noise N_TRIALS times, each time take the max
        noisy_qs   = Q_TRUE + np.random.normal(0, sigma, size=(N_TRIALS, len(Q_TRUE)))
        expected_max_noisy = noisy_qs.max(axis=1).mean()
        overestimation = expected_max_noisy - TRUE_MAX

    overestimations.append(overestimation)
    print(f"sigma={sigma:.1f}: E[max(Q+noise)]={TRUE_MAX + overestimation:.3f}, "
          f"true max={TRUE_MAX:.1f}, overestimation={overestimation:.3f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sigmas, overestimations, 'o-', color='tomato', linewidth=2.5, markersize=9)
ax.fill_between(sigmas, 0, overestimations, color='tomato', alpha=0.15)
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Noise Standard Deviation (sigma)', fontsize=12)
ax.set_ylabel('Overestimation: E[max(Q+noise)] - max(Q)', fontsize=12)
ax.set_title('Max Operator Overestimation vs. Noise Level\n'
             f'(10 Q-values, true max={TRUE_MAX}, N={N_TRIALS} trials)', fontsize=13)
ax.set_xticks(sigmas)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_522():
    assert len(overestimations) == len(sigmas), \
        f"Expected {len(sigmas)} overestimation values, got {len(overestimations)}"

    # At sigma=0, overestimation should be exactly 0
    assert abs(overestimations[0]) < 1e-9, \
        f"At sigma=0, overestimation should be 0, got {overestimations[0]:.6f}"

    # Overestimation should increase with sigma
    for i in range(len(sigmas) - 1):
        if sigmas[i] > 0:
            assert overestimations[i] < overestimations[i+1], (
                f"Overestimation should increase with sigma. "
                f"Got sigma={sigmas[i]}: {overestimations[i]:.3f}, "
                f"sigma={sigmas[i+1]}: {overestimations[i+1]:.3f}"
            )

    # At sigma=5, overestimation should be substantial
    assert overestimations[-1] > 1.0, (
        f"At sigma=5, expected overestimation > 1.0, got {overestimations[-1]:.3f}"
    )

    print("Exercise 5.2.2 passed!")
    print(f"Max overestimation (sigma=5): {overestimations[-1]:.3f}")

test_exercise_522()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **Q-value overestimation is systematic**: the max operator over noisy estimates always produces a positive bias. This bias grows with the noise level (i.e., early in training when estimates are poor).
2. **Double DQN eliminates overestimation** by decoupling selection (online net) from evaluation (target net). Since the two networks have different noise patterns, they partially cancel each other's overestimates.
3. **The code change is minimal**: only the target computation changes (two lines). Everything else — the replay buffer, network architecture, training loop — stays identical.
4. **Soft target updates** (Polyak averaging) provide a smoother alternative to hard copies. They are standard in actor-critic methods (DDPG, SAC, TD3) but work well for DQN too.
5. **Overestimation slows learning**, not just skews values: the agent that overestimates Q-values often pursues suboptimal actions because it thinks they are better than they are.

## What's Next

Module 6 introduces **Policy Gradient** methods — a fundamentally different approach that directly optimizes the policy without maintaining a Q-function. REINFORCE and Actor-Critic will show you how to handle continuous action spaces and why policy gradients are the foundation of modern RL (PPO, SAC, TD3).

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])